# Liu2024 — TWFB Shifting-Window / Filter-Band Diagnostics (starter)

A first-version, deliberately scoped tool to study **where in time and in which frequency band**
the motor-imagery signal is strongest in Liu2024, in the spirit of the **TWFB** (time-window +
filter-bank) search that feeds the paper's DGFMDRM baseline. See the companion note
`TWFB_DGFMDRM_Liu2024_Notes.md` for the conceptual background.

**What this notebook does:**
1. sweep shifted **time windows** (configurable start grid) × overlapping **filter bands**;
2. classify each (window, band) cell with **covariance + Minimum Distance to Riemannian Mean
   (MDM)** under the project's within-subject CV;
3. summarize the **best window/band per subject**;
4. **visualize** subject-level time/frequency sensitivity (heatmaps);
5. save artifacts in the same style as the other project notebooks.

**Scope / honesty (see Section 7 for the full breakdown):** this implements the **MDRM** half of
DGFMDRM — affine-invariant MDM via `pyriemann` when installed, with a dependency-free
**log-Euclidean MDM** fallback. It does **not** implement **discriminant geodesic filtering
(DGF)**, and the band/window grids and the selection criterion are reasonable defaults, not a
verified reproduction of Liu2024's exact protocol.

# 1. Setup

In [7]:
import copy, os, re, sys, json, math, hashlib, random, builtins, platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.io import loadmat
from scipy import signal
from scipy.linalg import eigh

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print(f"[setup] matplotlib unavailable -> plots skipped: {exc}")


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, Subset
    HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False
    Dataset = object
    print(f"[setup] torch unavailable -> EEGNet path skipped: {exc}")



try:
    import mne
    mne.set_log_level("WARNING")
    HAVE_MNE = True
except Exception as exc:
    HAVE_MNE = False
    print(f"[setup] mne unavailable -> data loading skipped: {exc}")

try:
    from pyriemann.estimation import Covariances
    from pyriemann.classification import MDM
    HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False
    print(f"[setup] pyriemann unavailable -> using log-Euclidean MDM fallback: {exc}")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("deps:", dict(mne=HAVE_MNE, pyriemann=HAVE_PYRIEMANN, mpl=HAVE_MPL),
      "| MDM backend:", "pyriemann (affine-invariant)" if HAVE_PYRIEMANN else "log-Euclidean fallback")


[2026-06-14 09:41:37] deps: {'mne': True, 'pyriemann': True, 'mpl': True} | MDM backend: pyriemann (affine-invariant)


# 2. Configuration

## 2.1 Channel defaults (carried verbatim)

In [8]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 2.2 CONFIG

Reference CONFIG carried verbatim, then extended with the TWFB sweep settings. **Note one
deliberate, documented change for this analysis:** the window is made *seconds-first and
shorter* (`target_window_samples=None`, `target_window_s=2.0`) so it can be **shifted** across
the imagery period — a fixed 4.2 s window cannot slide within 0–4 s. This is specific to the
shifting-window study and is not a comparability run against the reference.

In [9]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # braindecode on-the-fly augmentation.
    # Applied to the TRAINING iterator ONLY (via AugmentedDataLoader), so the
    # validation split skorch carves out internally is never augmented -> no leakage.
    # There is no fixed "number of augmented samples": the model sees a freshly
    # augmented view of the train fold every epoch. Control INTENSITY with each
    # transform's "probability" (how often it fires) and its magnitude params.
    # Ready-to-use configs are in the markdown cell just below CONFIG.
    # ------------------------------------------------------------------
    # "augmentation": {
    #     "enabled": False,        # master switch
    #     "name": "none",          # label, saved with artifacts
    #     "random_state": 2026,
    #     # each entry: {"name": <transform>, "probability": 0..1, <transform params>}
    #     "transforms": [],
    # },

    "augmentation": {
        "enabled": True,
        "name": "time_mask",
        "random_state": 2026,
        "transforms": [
        {
            "mask_len_samples": 64,
            "name": "smooth_time_mask",
            "probability": 0.5
        }
        ]
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

# ------------------------------------------------------------------ #
#  TWFB shifting-window diagnostics additions.                       #
# ------------------------------------------------------------------ #
CONFIG["experiment_name"] = "twfb_shifting_window_diagnostics"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-twfb-shifting-window-diagnostics")
CONFIG["config_note"] = ("TWFB-style time-window x filter-band sweep with covariance + MDM (MDRM). "
                         "DGF not implemented. Shorter seconds-first window so it can shift across 0-4 s.")

# Shorter, shiftable window (seconds-first). DELIBERATE change from the reference 4.2 s / 537.
CONFIG["target_window_samples"] = None
CONFIG["target_window_s"] = 2.0
CONFIG["augmentation"] = {"enabled": False, "name": "none", "random_state": 2026, "transforms": []}

# TWFB sweep grids.
CONFIG["twfb"] = {
    # time-window START offsets (s) from trial start; window length = CONFIG["target_window_s"].
    # VERIFY against the dataset's cue timing how these map onto the 0-4 s imagery period.
    "window_start_grid_s": [0.0, 0.5, 1.0, 1.5, 2.0],
    # overlapping filter bank tiling ~8-30 Hz (4 Hz wide, 2 Hz step) + a broad mu+beta band.
    "filter_bands_hz": [[8, 12], [10, 14], [12, 16], [14, 18], [16, 20],
                         [18, 22], [20, 24], [22, 26], [24, 28], [26, 30], [8, 30]],
    "cov_estimator": "oas",        # used by pyriemann path
    "cov_shrinkage": 1e-3,         # used by log-Euclidean fallback (regularize toward identity)
    "max_subjects": None,          # None = all
    "selection_metric": "balanced_accuracy",
}

print(f"Experiment: {CONFIG['experiment_name']}")
print(f"window length: {CONFIG['target_window_s']}s | starts: {CONFIG['twfb']['window_start_grid_s']}")
print(f"bands: {CONFIG['twfb']['filter_bands_hz']}")


[2026-06-14 09:41:37] Experiment: twfb_shifting_window_diagnostics
[2026-06-14 09:41:37] window length: 2.0s | starts: [0.0, 0.5, 1.0, 1.5, 2.0]
[2026-06-14 09:41:37] bands: [[8, 12], [10, 14], [12, 16], [14, 18], [16, 20], [18, 22], [20, 24], [22, 26], [24, 28], [26, 30], [8, 30]]


## 2.3 Derived constants (carried verbatim)

In [10]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


[2026-06-14 09:41:37] Effective Liu2024 Source MAT settings:
[2026-06-14 09:41:37]   Experiment:                        twfb_shifting_window_diagnostics
[2026-06-14 09:41:37]   Note:                              TWFB-style time-window x filter-band sweep with covariance + MDM (MDRM). DGF not implemented. Shorter seconds-first window so it can shift across 0-4 s.
[2026-06-14 09:41:37]   Channels:                          29
[2026-06-14 09:41:37]   Channel names:                     ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']
[2026-06-14 09:41:37]   Source sfreq:                      500 Hz
[2026-06-14 09:41:37]   Effective sfreq:                   128.0 Hz
[2026-06-14 09:41:37]   MI window start / samples:         1.5 s / 256
[2026-06-14 09:41:37]   Effective window duration:         2.0000 s
[2026-06-14 09:41:37]   Evaluation mode:               

## 2.4 Artifacts and logging (carried verbatim)

In [11]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-14 09:41:37] Run ID:     20260614_0941_8720608a
[2026-06-14 09:41:37] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-shifting-window-diagnostics/20260614_0941_8720608a
[2026-06-14 09:41:37] Config:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-shifting-window-diagnostics/20260614_0941_8720608a/config.json


## 2.5 Reproducibility (carried verbatim)

In [12]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-14 09:41:37] Using device: mps
[2026-06-14 09:41:38] Seed initialized: 2026


# 3. Carried machinery (verbatim)
Data loading, preprocessing, dataset classes, JSON-safe helpers, and the evaluation-split
builder, unchanged from the reference.

### 3.1 Data-loading helpers

In [13]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


### 3.2 Preprocessing pipeline

In [14]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),  # type: ignore
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "crop_fixed_mi_window",
        "start_s": config.get("mi_window_start_s"),
        "target_window_samples": config.get("target_window_samples"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    pipeline.append({
        "stage": "split",
        "name": "fold_safe_normalization",
        "mode": config.get("normalization_mode", "none"),
        "enabled": not _none_like(config.get("normalization_mode", "none")),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    start_sample = int(round(float(config["mi_window_start_s"]) * effective_sfreq))
    window_samples = int(config["target_window_samples"]) if config.get("target_window_samples") is not None else int(round(float(config["target_window_s"]) * effective_sfreq))
    stop_sample = start_sample + window_samples

    if stop_sample > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
            f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
        )

    X_win = X_rs[:, :, start_sample:stop_sample]
    runtime_steps.append(f"crop fixed window samples [{start_sample}:{stop_sample}]")

    y = labels_to_zero_based(labels)
    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


[2026-06-14 09:41:38] Configured preprocessing pipeline:
[2026-06-14 09:41:38]   - [ON] source::select_eeg_channels | - select Liu EEG channels, drop CPz reference, EOG, and marker before model input
[2026-06-14 09:41:38]   - [off] source::demean | mode=none | baseline_window_s=[0.0, 2.0]
[2026-06-14 09:41:38]   - [off] source::detrend | mode=none
[2026-06-14 09:41:38]   - [off] source::eog_correction | mode=none
[2026-06-14 09:41:38]   - [off] source::artifact_clipping | mode=none | percentile=99.5
[2026-06-14 09:41:38]   - [ON] mne_raw::reference | timing=before_resample_filter | mode=average
[2026-06-14 09:41:38]   - [ON] mne_raw::resample | sfreq=128
[2026-06-14 09:41:38]   - [ON] mne_raw::bandpass_filter | l_freq=0.5 | h_freq=40.0 | method=fir | phase=zero | fir_design=firwin
[2026-06-14 09:41:38]   - [off] mne_raw::notch_filter | timing=after_bandpass
[2026-06-14 09:41:38]   - [ON] window::crop_fixed_mi_window | start_s=1.5
[2026-06-14 09:41:38]   - [off] window::bad_trial_reject

### 3.3 Dataset classes

In [15]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


### 3.4 JSON-safe helpers

In [16]:
def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()


### 3.5 Evaluation-split builder (same protocol/seeds as reference)

In [17]:
def make_evaluation_splits(y, n_classes):
    """Create subject-level evaluation splits from CONFIG.

    Supported modes:
      - stratified_kfold: S-JEPA-style 5-fold within-subject CV
      - liu2024_repeated_60_40: 10 repeated stratified 60/40 train/test splits
      - repeated_stratified_split: generic repeated stratified holdout
    """
    y = np.asarray(y, dtype=np.int64)
    counts = np.bincount(y, minlength=n_classes)
    indices = np.arange(len(y))
    mode = str(CONFIG.get("evaluation_mode", "stratified_kfold")).lower()

    if mode == "stratified_kfold":
        n_folds = int(CONFIG.get("cv_folds", 5))
        if counts.min() < n_folds:
            raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}.")

        splitter = StratifiedKFold(
            n_splits=n_folds,
            shuffle=True,
            random_state=int(CONFIG.get("cv_random_state", CONFIG.get("seed", 2026))),
        )
        splits = []
        for fold_id, (train_idx, test_idx) in enumerate(splitter.split(indices, y), start=1):
            if CONFIG.get("assert_balanced_folds", True) and np.all(counts % n_folds == 0):
                expected_test = (counts // n_folds).astype(int)
                expected_train = (counts - expected_test).astype(int)
                train_counts = np.bincount(y[train_idx], minlength=n_classes)
                test_counts = np.bincount(y[test_idx], minlength=n_classes)
                assert np.array_equal(train_counts, expected_train), (
                    f"Unexpected train class counts for fold {fold_id}: {train_counts.tolist()} != {expected_train.tolist()}"
                )
                assert np.array_equal(test_counts, expected_test), (
                    f"Unexpected test class counts for fold {fold_id}: {test_counts.tolist()} != {expected_test.tolist()}"
                )
            splits.append({
                "split_id": int(fold_id),
                "fold_id": int(fold_id),
                "idx_train": train_idx,
                "idx_test": test_idx,
                "evaluation_mode": mode,
                "evaluation_protocol": f"{n_folds}-fold stratified within-subject CV",
                "n_total_splits": int(n_folds),
            })
        return splits

    if mode in ("liu2024_repeated_60_40", "repeated_stratified_split"):
        n_repeats = int(CONFIG.get("n_repeats", 10))
        test_size = float(CONFIG.get("test_size", 0.40))
        splitter = StratifiedShuffleSplit(
            n_splits=n_repeats,
            test_size=test_size,
            random_state=int(CONFIG.get("split_random_state", CONFIG.get("seed", 2026))),
        )
        splits = []
        for split_id, (train_idx, test_idx) in enumerate(splitter.split(indices, y), start=1):
            train_counts = np.bincount(y[train_idx], minlength=n_classes)
            test_counts = np.bincount(y[test_idx], minlength=n_classes)
            if CONFIG.get("assert_balanced_folds", True):
                if train_counts.min() < 1 or test_counts.min() < 1:
                    raise ValueError(
                        f"Invalid stratified holdout split {split_id}: train_counts={train_counts.tolist()}, "
                        f"test_counts={test_counts.tolist()}"
                    )
            protocol = "Liu2024-style 10 repeated stratified 60/40 train/test splits" if mode == "liu2024_repeated_60_40" else f"{n_repeats} repeated stratified holdout splits, test_size={test_size}"
            splits.append({
                "split_id": int(split_id),
                "fold_id": int(split_id),
                "idx_train": train_idx,
                "idx_test": test_idx,
                "evaluation_mode": mode,
                "evaluation_protocol": protocol,
                "n_total_splits": int(n_repeats),
                "test_size": float(test_size),
            })
        return splits

    raise ValueError(f"Unsupported evaluation_mode={CONFIG.get('evaluation_mode')}")

# Backward compatibility.
def make_fold_splits(y, n_folds, n_classes):
    old_cv_folds = CONFIG.get("cv_folds")
    old_mode = CONFIG.get("evaluation_mode")
    CONFIG["evaluation_mode"] = "stratified_kfold"
    CONFIG["cv_folds"] = int(n_folds)
    try:
        return make_evaluation_splits(y, n_classes)
    finally:
        CONFIG["cv_folds"] = old_cv_folds
        CONFIG["evaluation_mode"] = old_mode


### 3.6 Lightweight metrics + collapse diagnostics
Small, faithful reimplementations of the reference's two array-based diagnostics, so this
notebook needs neither torch nor the S-JEPA classifier cell.

In [18]:
def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred))}

def compute_collapse_diagnostics(y_pred, n_classes):
    h = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    n = int(h.sum()); ratio = float(h.max() / n) if n else 0.0
    thr = float(CONFIG.get("collapse_threshold", 0.90))
    return {"prediction_histogram": h.tolist(), "collapse_ratio": ratio,
            "collapse_threshold": thr, "collapse_flag": bool(ratio >= thr),
            "majority_predicted_class": int(h.argmax()) if n else None}


# 4. Load data (carried verbatim)
Builds `SUBJECT_ARRAYS`-equivalent structures. Requires the Liu2024 `.mat` files and MNE.
Because the TWFB sweep re-windows at several start offsets, we keep the **raw** per-subject
arrays (pre-window) and re-run preprocessing per start.

In [19]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)

if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError(
        "Could not find Liu2024 source .mat files. "
    )

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

# Write a structure preview for the first subject. This makes it clear whether
# the local files expose top-level rawdata/labels or an `eeg` struct.
preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(EFFECTIVE_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
preprocessing_steps_first_subject = None

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial, preprocessing_steps, preprocessing_stats = preprocess_subject_configurable(X_raw, y_raw, sid)

    if preprocessing_steps_first_subject is None:
        preprocessing_steps_first_subject = preprocessing_steps

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "source_unit": CONFIG["source_unit"],
        "final_model_unit": CONFIG["final_model_unit"],
        "demean_mode": CONFIG["demean_mode"],
        "reference_mode": CONFIG["reference_mode"],
        "reference_timing": CONFIG["reference_timing"],
        "resample": bool(CONFIG["resample"]),
        "effective_sfreq": float(EFFECTIVE_SFREQ),
        "filter_enabled": bool(CONFIG["filter_enabled"]),
        "filter_low": CONFIG.get("filter_low"),
        "filter_high": CONFIG.get("filter_high"),
        "filter_method": CONFIG.get("filter_method"),
        "mi_window_start_s": float(CONFIG["mi_window_start_s"]),
        "eog_correction": CONFIG.get("eog_correction"),
        "artifact_clip_mode": CONFIG.get("artifact_clip_mode"),
        "reject_bad_trials": bool(CONFIG.get("reject_bad_trials", False)),
        "normalization_mode": CONFIG.get("normalization_mode"),
        **preprocessing_stats,
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

if preprocessing_steps_first_subject is not None:
    print("Preprocessing steps used:")
    for step in preprocessing_steps_first_subject:
        print(f"  - {step}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


[2026-06-14 09:41:38] Source extract dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata
[2026-06-14 09:41:38] Found 50 .mat files
[2026-06-14 09:41:38] MAT structure preview saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-shifting-window-diagnostics/20260614_0941_8720608a/mat_structure_preview_first_subject.csv
[2026-06-14 09:41:44] Subjects loaded: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[2026-06-14 09:41:44] Subject inventory saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-shifting-window-diagnostics/20260614_0941_8720608a/subject_inventory.csv
[2026-06-14 09:41:59] X_ALL shape: (2000, 29, 256) | Y_ALL counts: [1000, 1000]
[2026-0

,subject_id,n_windows,class_counts,preprocessed_shape,resampled_samples_per_trial,crop_start_sample,crop_stop_sample,target_window_samples,effective_window_duration_s,source_unit,...,reject_bad_trials,normalization_mode,artifact_clip_applied,artifact_clip_threshold,n_trials_before_reject,n_trials_after_reject,n_rejected_trials,rejection_skipped,pipeline_plan,runtime_steps
0,1,40,"[20, 20]","(40, 29, 256)",1024,192,448,256,2.0,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
1,2,40,"[20, 20]","(40, 29, 256)",1024,192,448,256,2.0,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
2,3,40,"[20, 20]","(40, 29, 256)",1024,192,448,256,2.0,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
3,4,40,"[20, 20]","(40, 29, 256)",1024,192,448,256,2.0,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
4,5,40,"[20, 20]","(40, 29, 256)",1024,192,448,256,2.0,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...


In [20]:
# Cache raw (pre-preprocessing) arrays so the sweep can re-window at each start offset.
RAW_SUBJECTS = {}
for item in subjects_df.to_dict("records"):
    sid = str(int(item["subject_id"]))
    Xr, yr, _, _ = load_subject_mat(Path(item["path"]))
    RAW_SUBJECTS[sid] = (Xr, yr)
print(f"Cached raw arrays for {len(RAW_SUBJECTS)} subjects.")


[2026-06-14 09:42:05] Cached raw arrays for 50 subjects.


# 5. Covariance + Minimum Distance to Riemannian Mean (the MDRM classifier)

Per fold: estimate one covariance matrix per trial from the band-filtered window, then classify
by the nearest **Riemannian class-mean** covariance. With `pyriemann` this is the
affine-invariant MDM; otherwise a **log-Euclidean** fallback (mean in the matrix-log domain,
Frobenius distance). Fold-safe: class means come from the training split only.

In [21]:
def _bandpass(x, lo, hi, sfreq, order=4):
    nyq = 0.5 * sfreq
    lo_n, hi_n = max(lo / nyq, 1e-4), min(hi / nyq, 0.999)
    if hi_n <= lo_n:
        return x
    b, a = signal.butter(order, [lo_n, hi_n], btype="band")
    return signal.filtfilt(b, a, x, axis=-1)

def _logm_spd(C):
    """Matrix log of a symmetric PD matrix via eigendecomposition."""
    w, V = eigh(C)
    w = np.clip(w, 1e-12, None)
    return (V * np.log(w)) @ V.T

def _trial_covariances(X, shrinkage):
    """X: (n_trials, n_chans, n_times) -> (n_trials, n_chans, n_chans) regularized SPD."""
    n_tr, n_ch, n_t = X.shape
    covs = np.empty((n_tr, n_ch, n_ch))
    eye = np.eye(n_ch)
    for i in range(n_tr):
        Xi = X[i] - X[i].mean(axis=1, keepdims=True)
        C = (Xi @ Xi.T) / max(n_t - 1, 1)
        tr = np.trace(C) / n_ch
        covs[i] = (1 - shrinkage) * C + shrinkage * tr * eye   # shrink toward scaled identity
    return covs

def mdm_fold(Xtr, ytr, Xte, n_classes, cov_estimator="oas", shrinkage=1e-3):
    """Return (y_pred, pseudo_probs). Affine-invariant MDM via pyriemann, else log-Euclidean."""
    if HAVE_PYRIEMANN:
        cov = Covariances(estimator=cov_estimator)
        Ctr, Cte = cov.fit_transform(Xtr), cov.transform(Xte)
        clf = MDM().fit(Ctr, ytr)
        try:
            probs = np.asarray(clf.predict_proba(Cte), dtype=float)
        except Exception:
            probs = None
        return np.asarray(clf.predict(Cte), dtype=int), probs
    # ---- log-Euclidean fallback ----
    Ctr = _trial_covariances(Xtr, shrinkage)
    Cte = _trial_covariances(Xte, shrinkage)
    logs_tr = np.stack([_logm_spd(C) for C in Ctr])
    logs_te = np.stack([_logm_spd(C) for C in Cte])
    class_means = []
    classes = np.arange(n_classes)
    for c in classes:
        m = (ytr == c)
        class_means.append(logs_tr[m].mean(axis=0) if m.any() else np.zeros_like(logs_tr[0]))
    class_means = np.stack(class_means)
    dists = np.stack([np.linalg.norm(logs_te - cm, axis=(1, 2)) for cm in class_means], axis=1)
    y_pred = dists.argmin(axis=1).astype(int)
    # pseudo-probabilities from negative distances (for collapse/confidence diagnostics only)
    z = -dists
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    probs = e / np.maximum(e.sum(axis=1, keepdims=True), 1e-12)
    return y_pred, probs

def evaluate_cell(X, y, band, n_classes, twfb_cfg):
    """One (window, band) cell for one subject: band-filter, then within-subject CV with MDM."""
    Xb = _bandpass(X, band[0], band[1], EFFECTIVE_SFREQ)
    splits = make_evaluation_splits(y, n_classes)
    accs, bals, hists, collapses = [], [], [], []
    for s in splits:
        Xtr, ytr = Xb[s["idx_train"]], y[s["idx_train"]]
        Xte, yte = Xb[s["idx_test"]], y[s["idx_test"]]
        y_pred, _ = mdm_fold(Xtr.astype(np.float64), ytr, Xte.astype(np.float64), n_classes,
                             twfb_cfg.get("cov_estimator", "oas"), twfb_cfg.get("cov_shrinkage", 1e-3))
        m = compute_classification_metrics(yte, y_pred)
        accs.append(m["accuracy"]); bals.append(m["balanced_accuracy"])
        col = compute_collapse_diagnostics(y_pred, n_classes)
        hists.append(col["prediction_histogram"]); collapses.append(col["collapse_flag"])
    hist_tot = np.sum(hists, axis=0).tolist()
    return {"mean_accuracy": float(np.mean(accs)), "mean_balanced_accuracy": float(np.mean(bals)),
            "std_balanced_accuracy": float(np.std(bals)), "collapse_rate": float(np.mean(collapses)),
            "prediction_histogram_total": hist_tot, "n_folds": len(splits)}


# 6. Run the time-window × filter-band sweep

For each window start (re-preprocess) and each band (re-filter), evaluate every subject with the
MDM classifier. Produces a long-format table `(subject, start_s, band, balanced_accuracy, ...)`,
the per-subject best (window, band), and per-subject + group time/frequency heatmaps.

In [22]:
twfb = CONFIG["twfb"]
GRID_ROWS = []
if not HAVE_MNE:
    print("MNE unavailable -> cannot preprocess; sweep skipped.")
else:
    sids = sorted(RAW_SUBJECTS.keys(), key=_sort_subject_key)
    if twfb.get("max_subjects"):
        sids = sids[:int(twfb["max_subjects"])]
    band_labels = [f"{lo}-{hi}" for lo, hi in twfb["filter_bands_hz"]]
    saved_start = CONFIG["mi_window_start_s"]
    resampled_len = int(round(LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL * EFFECTIVE_SFREQ / LIU_SOURCE_SFREQ))
    try:
        for start_s in twfb["window_start_grid_s"]:
            stop_sample = int(round(start_s * EFFECTIVE_SFREQ)) + int(WINDOW_SAMPLES)
            if stop_sample > resampled_len:
                print(f"  start={start_s}s skipped (window exceeds trial: {stop_sample}>{resampled_len})")
                continue
            CONFIG["mi_window_start_s"] = float(start_s)
            # preprocess each subject once at this start, then sweep bands in-memory
            for sid in sids:
                Xr, yr = RAW_SUBJECTS[sid]
                Xw, yy, _, _, _ = preprocess_subject_configurable(Xr, yr, int(sid))
                Xw = Xw.astype(np.float64); yy = yy.astype(int)
                for (lo, hi), blab in zip(twfb["filter_bands_hz"], band_labels):
                    cell = evaluate_cell(Xw, yy, (lo, hi), TARGET_N_CLASSES, twfb)
                    GRID_ROWS.append({"subject_id": sid, "window_start_s": float(start_s),
                                      "band": blab, "band_lo": lo, "band_hi": hi,
                                      **cell})
                print(f"  start={start_s}s subject {sid}: swept {len(band_labels)} bands")
    finally:
        CONFIG["mi_window_start_s"] = saved_start  # restore (no silent change)

GRID_DF = pd.DataFrame(GRID_ROWS)
if not GRID_DF.empty:
    GRID_DF.to_csv(ARTIFACT_DIR / "twfb_grid_long.csv", index=False)
    print(f"\nGrid: {len(GRID_DF)} (subject x window x band) cells saved to twfb_grid_long.csv")


[2026-06-14 09:42:08]   start=0.0s subject 1: swept 11 bands
[2026-06-14 09:42:10]   start=0.0s subject 2: swept 11 bands
[2026-06-14 09:42:13]   start=0.0s subject 3: swept 11 bands
[2026-06-14 09:42:15]   start=0.0s subject 4: swept 11 bands
[2026-06-14 09:42:18]   start=0.0s subject 5: swept 11 bands
[2026-06-14 09:42:20]   start=0.0s subject 6: swept 11 bands
[2026-06-14 09:42:22]   start=0.0s subject 7: swept 11 bands
[2026-06-14 09:42:25]   start=0.0s subject 8: swept 11 bands
[2026-06-14 09:42:27]   start=0.0s subject 9: swept 11 bands
[2026-06-14 09:42:29]   start=0.0s subject 10: swept 11 bands
[2026-06-14 09:42:32]   start=0.0s subject 11: swept 11 bands
[2026-06-14 09:42:34]   start=0.0s subject 12: swept 11 bands
[2026-06-14 09:42:36]   start=0.0s subject 13: swept 11 bands
[2026-06-14 09:42:38]   start=0.0s subject 14: swept 11 bands
[2026-06-14 09:42:41]   start=0.0s subject 15: swept 11 bands
[2026-06-14 09:42:44]   start=0.0s subject 16: swept 11 bands
[2026-06-14 09:42

# 7. Best window/band per subject + sensitivity maps

In [23]:
BEST_PER_SUBJECT = None
if not GRID_DF.empty:
    idx = GRID_DF.groupby("subject_id")["mean_balanced_accuracy"].idxmax()
    BEST_PER_SUBJECT = GRID_DF.loc[idx, ["subject_id", "window_start_s", "band",
                                         "mean_balanced_accuracy", "std_balanced_accuracy",
                                         "collapse_rate"]].sort_values("subject_id").reset_index(drop=True)
    print("Best (window, band) per subject by balanced accuracy:")
    try:
        from IPython.display import display
        display(BEST_PER_SUBJECT.round(3))
    except Exception:
        print(BEST_PER_SUBJECT.round(3).to_string(index=False))
    BEST_PER_SUBJECT.to_csv(ARTIFACT_DIR / "twfb_best_per_subject.csv", index=False)

    # group-mean grid (window x band) + per-subject small multiples
    group_mean = GRID_DF.groupby(["window_start_s", "band"])["mean_balanced_accuracy"].mean().reset_index()
    group_pivot = group_mean.pivot(index="window_start_s", columns="band", values="mean_balanced_accuracy")
    # keep band column order as configured
    band_labels = [f"{lo}-{hi}" for lo, hi in CONFIG["twfb"]["filter_bands_hz"]]
    group_pivot = group_pivot.reindex(columns=[b for b in band_labels if b in group_pivot.columns])
    group_pivot.to_csv(ARTIFACT_DIR / "twfb_group_mean_grid.csv")
    print("\nGroup-mean balanced accuracy (window x band):")
    print(group_pivot.round(3).to_string())

    if HAVE_MPL:
        # group-mean heatmap
        fig, ax = plt.subplots(figsize=(1.0 * group_pivot.shape[1] + 3, 0.6 * group_pivot.shape[0] + 2))
        im = ax.imshow(group_pivot.values, aspect="auto", vmin=0.4, vmax=0.8, cmap="viridis")
        ax.set_xticks(range(group_pivot.shape[1])); ax.set_xticklabels(group_pivot.columns, rotation=45, ha="right")
        ax.set_yticks(range(group_pivot.shape[0])); ax.set_yticklabels(group_pivot.index)
        ax.set_xlabel("filter band (Hz)"); ax.set_ylabel("window start (s)")
        ax.set_title("Group-mean balanced accuracy (TWFB grid)")
        fig.colorbar(im, ax=ax, label="balanced accuracy")
        fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "twfb_group_mean_heatmap.png", dpi=160); plt.close(fig)
        print(f"Saved: {ARTIFACT_DIR / 'twfb_group_mean_heatmap.png'}")

        # per-subject small multiples
        sids = sorted(GRID_DF["subject_id"].unique(), key=_sort_subject_key)
        ncol = min(4, len(sids)); nrow = int(np.ceil(len(sids) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.6 * nrow), squeeze=False)
        for k, sid in enumerate(sids):
            ax = axes[k // ncol][k % ncol]
            sub = GRID_DF[GRID_DF["subject_id"] == sid]
            piv = sub.pivot(index="window_start_s", columns="band", values="mean_balanced_accuracy")
            piv = piv.reindex(columns=[b for b in band_labels if b in piv.columns])
            im = ax.imshow(piv.values, aspect="auto", vmin=0.4, vmax=0.9, cmap="viridis")
            ax.set_title(f"subject {sid}", fontsize=9)
            ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=90, fontsize=5)
            ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=6)
        for k in range(len(sids), nrow * ncol):
            axes[k // ncol][k % ncol].set_visible(False)
        fig.suptitle("Per-subject time/frequency sensitivity (balanced accuracy)")
        fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "twfb_subject_sensitivity_maps.png", dpi=150); plt.close(fig)
        print(f"Saved: {ARTIFACT_DIR / 'twfb_subject_sensitivity_maps.png'}")
else:
    print("No grid results (need data + MNE).")


[2026-06-14 09:51:08] Best (window, band) per subject by balanced accuracy:


,subject_id,window_start_s,band,mean_balanced_accuracy,std_balanced_accuracy,collapse_rate
0,1,0.0,16-20,0.600,0.242,0.0
1,10,2.0,24-28,0.725,0.184,0.0
2,11,1.5,8-12,0.700,0.170,0.0
3,12,2.0,18-22,0.700,0.187,0.0
4,13,2.0,22-26,0.575,0.100,0.0
5,14,1.0,14-18,0.700,0.150,0.2
6,15,1.0,18-22,0.725,0.094,0.0
7,16,0.5,14-18,0.675,0.187,0.0
8,17,0.0,10-14,0.650,0.146,0.0
9,18,1.0,14-18,0.625,0.112,0.0



[2026-06-14 09:51:09] Group-mean balanced accuracy (window x band):
[2026-06-14 09:51:09] band             8-12  10-14  12-16  14-18  16-20  18-22  20-24  22-26  24-28  26-30   8-30
window_start_s                                                                             
0.0             0.484  0.480  0.462  0.472  0.461  0.456  0.477  0.474  0.461  0.447  0.452
0.5             0.495  0.482  0.490  0.470  0.480  0.480  0.496  0.484  0.464  0.460  0.480
1.0             0.488  0.488  0.476  0.468  0.480  0.475  0.489  0.495  0.455  0.438  0.467
1.5             0.515  0.493  0.477  0.486  0.492  0.467  0.470  0.476  0.470  0.444  0.469
2.0             0.488  0.488  0.490  0.480  0.480  0.472  0.507  0.472  0.482  0.464  0.473
[2026-06-14 09:51:09] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-shifting-window-diagnostics/20260614_0941_8720608a/twfb_group_mean_heatmap.png
[2026-06-14 09:51:11] Saved: /Users/vadim/Documents/S

## 8. Save run metadata

In [24]:
run_metadata = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG.get("experiment_name"), "config_note": CONFIG.get("config_note"),
    "mdm_backend": "pyriemann_affine_invariant" if HAVE_PYRIEMANN else "log_euclidean_fallback",
    "twfb": CONFIG["twfb"], "window_len_s": CONFIG["target_window_s"],
    "window_samples": int(WINDOW_SAMPLES), "effective_sfreq": EFFECTIVE_SFREQ,
    "preprocessing_config": PREPROCESSING_CONFIG, "evaluation_config": EVALUATION_CONFIG,
    "channel_names": list(EEG_CHANNEL_NAMES), "seed": int(BASE_SEED),
    "implemented": "shifting time-window x filter-band sweep; covariance + MDM (MDRM)",
    "not_implemented": "discriminant geodesic filtering (DGF); exact Liu2024 bands/windows/selection protocol",
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Saved run metadata to: {ARTIFACT_DIR / 'run_metadata.json'}")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {p.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


[2026-06-14 09:51:11] Saved run metadata to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-shifting-window-diagnostics/20260614_0941_8720608a/run_metadata.json
[2026-06-14 09:51:11]   - config.json
[2026-06-14 09:51:11]   - mat_structure_preview_first_subject.csv
[2026-06-14 09:51:11]   - run.log
[2026-06-14 09:51:11]   - run_metadata.json
[2026-06-14 09:51:11]   - subject_inventory.csv
[2026-06-14 09:51:11]   - twfb_best_per_subject.csv
[2026-06-14 09:51:11]   - twfb_grid_long.csv
[2026-06-14 09:51:11]   - twfb_group_mean_grid.csv
[2026-06-14 09:51:11]   - twfb_group_mean_heatmap.png
[2026-06-14 09:51:11]   - twfb_subject_sensitivity_maps.png
[2026-06-14 09:51:11]   - window_counts_by_subject.csv


# 9. How to read this, and what is/ isn't reproduced

**Reading the maps.** Each subject's heatmap is balanced accuracy over window start (rows) ×
filter band (columns). Bright cells mark the time/frequency region where left-vs-right MI is
most separable for that subject. Patterns to look for:
- a **band peak** in mu (8–12) and/or beta (13–30) — the expected MI signature;
- a **time peak** at a particular window start — where ERD/ERS is strongest after the cue;
- subjects whose map is **uniformly near chance** — candidate "no decodable MI" subjects, which
  should line up with the hard subjects flagged by the supervised-diagnostics notebook.

**How this feeds S-JEPA.** Use each subject's (and the group's) best window/band to set S-JEPA's
input window and pre-filter band; compare S-JEPA embeddings against these Riemannian features
under a common linear probe; and consider fusing them. (See `TWFB_DGFMDRM_Liu2024_Notes.md` §5.)

**Faithfully implemented here**
- Shifting time-window × overlapping filter-bank sweep, project-standard preprocessing/CV/artifacts.
- Covariance + Minimum Distance to Riemannian Mean (affine-invariant via pyriemann; log-Euclidean
  fallback otherwise). Fold-safe class means.
- Per-subject best window/band and time/frequency sensitivity visualizations.

**Approximated / simplified (not silent)**
- **No discriminant geodesic filtering (DGF).** This is the MDRM half of DGFMDRM only.
- Band grid, window grid, and the shorter 2 s shiftable window are reasonable defaults, not
  Liu2024's verified settings.
- **Selection bias caveat:** "best (window, band)" is chosen on the same within-subject CV that
  scores it, so per-subject bests are optimistically biased. For an unbiased estimate, wrap the
  window/band selection in an outer (nested) CV before quoting numbers.

**Remaining to reproduce Liu2024 faithfully**
- The DGF formulation and its integration with MDM; the paper's exact bands/overlap, window set,
  and 0–4 s alignment; the exact selection + (nested) CV protocol; and any multi-window/band
  feature fusion.